# IKG Lineage SQL Stitcher

Concatenate the ordered list of source SQL scripts (from `sandbox_prj_smart_insights.ikg_table_lineage_auto_refresh_temp`) into a single deployable file, pulling each script from the `ikg-dags` GitLab repository and annotating the sources.

In [ ]:
import logging

from ikg_lineage_sql_stitcher import (
    GitLabSQLFetcher,
    fetch_lineage_entries,
    fetch_profile_tables,
    fetch_profile_scripts,
    parse_insight_values,
    stitch_sql,
    render_stitched_text,
    write_output_file,
    write_modified_file,
    run_sql_script,
    preview_final_table,
    EXCLUDE_FOLDER,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
print("Stitcher helpers ready.")

In [ ]:
import os
from pathlib import Path
import getpass

import gitlab
import psycopg2
from psycopg2 import sql

print("Environment ready.")

In [ ]:
INSIGHT_RAW = input("Enter insight_type (comma-separated if needed): ").strip()
if not INSIGHT_RAW:
    raise ValueError("insight_type is required.")

INSIGHT_VALUES = parse_insight_values(INSIGHT_RAW)
if not INSIGHT_VALUES:
    raise ValueError("Please provide at least one insight_type.")

PROFILE_DATE = input("Enter profile date (YYYYMMDD): ").strip()
if not PROFILE_DATE:
    raise ValueError("profile date is required.")

DB_PASSWORD = getpass.getpass("Enter Password for DB User: ")
DB_CONFIG = {
    "host": "greenplum-rdsp.zur.swissbank.com",
    "port": "5432",
    "dbname": "gprdsp",
    "user": "ds_rdsp_dev",
    "password": DB_PASSWORD,
}
print("Captured credentials for Greenplum.")

In [ ]:
with psycopg2.connect(**DB_CONFIG) as conn:
    entries = fetch_lineage_entries(conn)
    profile_tables = fetch_profile_tables(conn, INSIGHT_VALUES)

print(f"Retrieved {len(entries)} lineage entries.")
print(f"Profile tables to append: {profile_tables}")
entries[:5]

In [ ]:
PRIVATE_TOKEN = getpass.getpass("Enter your private token: ")
exclude_folders = [folder for folder in EXCLUDE_FOLDER.split() if folder]
fetcher = GitLabSQLFetcher(private_token=PRIVATE_TOKEN, exclude_folders=exclude_folders)
print("GitLab fetcher initialized.")

In [ ]:
stitched_sql = stitch_sql(entries, fetcher)
profile_scripts = fetch_profile_scripts(fetcher, profile_tables)
if profile_scripts:
    stitched_sql.extend(profile_scripts)
stitched_text = render_stitched_text(stitched_sql)
print(f"Stitched {len(stitched_sql)} SQL files (including {len(profile_scripts)} profile scripts).")

In [ ]:
output_new = write_output_file(INSIGHT_RAW, stitched_text)
output_modified, modified_text = write_modified_file(INSIGHT_RAW, stitched_text, PROFILE_DATE)
print(f"Created files: {output_new} (original), {output_modified} (modified)")

run_sql_script(modified_text, DB_CONFIG)
preview_final_table(DB_CONFIG, profile_tables)
print("Execution complete and preview retrieved.")